# Compare datasets

The cross-dataset view: every screen's pooled results on one axis, so the
feature sections can be read against each other. Same shape as the old
`compare_featuresets.ipynb` at the repo root, but reading the hub's own runs.

One thing is simpler here. In the old notebook each project had its own copy of
`gpc`, so comparing them meant clearing `sys.modules` and re-importing the right
package for each dataset -- and getting that wrong produced a `KeyError` on
another project's target column. The hub has **one** engine reading every
bundle's config, so that whole mechanism is gone.

Nothing is fitted here. This reads the exports the per-dataset reports already
built.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.transforms import blended_transform_factory
from IPython.display import display

ROOT = Path.cwd().resolve()          # gp_collab_hub/
sys.path.insert(0, str(ROOT))

from gpc import figures
from gpc.config import SELECTED_SETS, bundle_group, bundle_name, read_json
from gpc.results import (MODEL_ORDER, load_results, method_label, prepare,
                         rank_features)

pd.set_option("display.width", 220, "display.max_columns", 80)

# ---- Editable ------------------------------------------------------------
# One entry per dataset to compare: the run folder it came back in, and any
# feature sections to leave out of the charts.
DATASETS = {
    "perera":   {"runs": "runs/perera_vmin_20260924",   "exclude_models": []},
    "ahneman":  {"runs": "runs/ahneman_vmin_20260924",  "exclude_models": []},
    "gesmundo": {"runs": "runs/gesmundo_vmin_20260924", "exclude_models": []},
}
REFRESH = False
# --------------------------------------------------------------------------

tables_by_dataset, info_by_dataset = {}, {}
for name, entry in DATASETS.items():
    bundle = ROOT / "datasets" / name / "inputs"
    export = prepare(ROOT / entry["runs"], bundle=bundle, refresh=REFRESH)
    tables, collection = load_results(export)
    prepared = read_json(bundle / "config.json")
    tables_by_dataset[name] = tables
    info_by_dataset[name] = {"label": bundle_name(prepared),
                             "group": bundle_group(prepared),
                             "n_groups": prepared["data"].get("expected_groups"),
                             "rows": prepared["data"]["expected_rows"]}
    print(f"{name:10s} {bundle_name(prepared):10s} complete={collection['complete']}  "
          f"{collection['completed_tasks']}/{collection['expected_tasks']} tasks")

## The combined table

Feature sections carry different keys per dataset (`catalyst_ohe` against
`ligand_ohe`) even though they play the same role, so they are relabelled onto
one shared axis. `gpc.figures.FEATURESET_LABELS` is the same map the saved
figures use, so a chart here and a chart in a results folder name things
identically.

In [ ]:
rows = []
for name, entry in DATASETS.items():
    summary = tables_by_dataset[name]["summary"]
    pooled = summary[summary.aggregation == "pooled_predictions"].copy()
    pooled = pooled[~pooled.model.isin(entry["exclude_models"])]
    pooled["dataset"] = name
    pooled["featureset"] = pooled["model"].map(figures.FEATURESET_LABELS).fillna(pooled["model"])
    rows.append(pooled)
combined = pd.concat(rows, ignore_index=True)

# The feature-set axis, DERIVED rather than hardcoded. Canonical order comes
# from results.MODEL_ORDER, mapped through the same display labels the saved
# figures use, and restricted to what these runs actually contain. A section
# added to config.SELECTED_SETS therefore appears here on its own -- hardcoding
# the list is what previously hid the two Vmin pairs even though the runs had
# them. Anything unrecognised is appended rather than dropped.
present = set(combined.featureset)
FEATURESET_ORDER = list(dict.fromkeys(
    figures.featureset_label(m) for m in MODEL_ORDER
    if figures.featureset_label(m) in present))
FEATURESET_ORDER += sorted(present - set(FEATURESET_ORDER))
print("feature sets on the axis:", FEATURESET_ORDER)

# A section a dataset never ran shows as a gap in the charts rather than a
# silently shifted bar, so say which ones up front.
gaps = {name: sorted(set(FEATURESET_ORDER)
                     - set(combined.loc[combined.dataset == name, "featureset"]))
        for name in DATASETS}
gaps = {name: missing for name, missing in gaps.items() if missing}
if gaps:
    print("Missing feature sets (gaps in the charts below):", gaps)

# When a screen has exactly 5 groups, LOLO is already a 5-fold partition, so
# `iid_stratified_5` and `kfold_stratified_5` are the SAME splitter and their
# columns below are identical by construction -- not a copy-paste error.
for name in DATASETS:
    part = combined[combined.dataset == name]
    pair = part.pivot_table(index="featureset", columns="method", values="r2")
    if {"iid_matched", "kfold"} <= set(pair.columns) and np.allclose(
            pair["iid_matched"], pair["kfold"], equal_nan=True):
        print(f"{name}: matched IID and 5-fold CV are the same splitter here "
              f"({info_by_dataset[name]['n_groups']} groups), so their columns agree exactly.")

display(combined.pivot_table(index=["dataset", "featureset"], columns="method",
                             values="r2", aggfunc="first")
                .reindex(FEATURESET_ORDER, level="featureset").round(4))

In [ ]:
# x positions for `bars_per_group` bars in each of `n_groups` groups. Bars sit
# edge-to-edge inside a group; groups are separated by group_gap -- the slight
# gap between datasets. Returns (positions, bounds): positions is one array of
# x-coordinates per group, bounds is the (left, right) span of each group, for
# draw_group_brackets below.
def grouped_bar_positions(n_groups, bars_per_group, bar_width=0.8, group_gap=0.9):
    positions, bounds, cursor = [], [], 0.0
    for _ in range(n_groups):
        group = cursor + bar_width * np.arange(bars_per_group)
        positions.append(group)
        bounds.append((group[0] - bar_width / 2, group[-1] + bar_width / 2))
        cursor = group[-1] + bar_width + group_gap
    return positions, bounds


# A square bracket under each (left, right) span in `bounds`, with labels[i]
# centered beneath it -- the per-dataset bracket the feature-set tick labels
# sit inside. y/tick are axes-fraction units while x stays in data units, so
# the bracket sits at a fixed visual depth regardless of the data's own scale.
def draw_group_brackets(ax, bounds, labels, y=-0.42, tick=0.035, fontsize=9):
    trans = blended_transform_factory(ax.transData, ax.transAxes)
    for (left, right), label in zip(bounds, labels):
        ax.plot([left, left, right, right], [y + tick, y, y, y + tick],
                color="0.3", linewidth=0.9, transform=trans, clip_on=False)
        ax.text((left + right) / 2, y - tick * 1.6, label, transform=trans,
                ha="center", va="top", fontsize=fontsize)


# ax.bar_label anchors each label just past the bar's far tip, which for a
# negative value lands below the x-axis -- off the plot, since the y limit is
# hard-floored at 0 below. This anchors at the value's own height when it is
# >= 0, and at the zero baseline when it is negative, so the number is still
# shown even though the bar itself is clipped at 0.
def label_bars(ax, bars, values, fmt="%.2f", fontsize=7, pad_points=2):
    for bar, value in zip(bars, values):
        if not np.isfinite(value):
            continue
        y = value if value >= 0 else 0.0
        ax.annotate(fmt % value, xy=(bar.get_x() + bar.get_width() / 2, y),
                    xytext=(0, pad_points), textcoords="offset points",
                    ha="center", va="bottom", fontsize=fontsize)


# Hard floor at 0, always -- a negative bar is clipped rather than shifting the
# axes down to show it; label_bars above still reports its value at the baseline.
def set_bar_ylim(ax, values, top_margin=1.15):
    hi = max(0.0, float(np.nanmax(values)))
    ax.set_ylim(0, hi * top_margin)


# The method palette is figures.STYLE's, so these charts and the saved
# per-dataset ones colour the same method the same way.
METHOD_COLORS = figures.STYLE["pooled"]["method_colors"]

# One distinguishable colour per feature set (colourblind-safe, Okabe-Ito).
# Built from FEATURESET_ORDER so a newly added section gets a colour instead of
# raising a KeyError; the five long-standing sections keep the colours they have
# always had, so a new chart still reads against an older one.
FEATURESET_FIXED = {
    "OHE": "#E69F00", "Selected-2": "#0072B2", "Selected-5": "#009E73",
    "PC top": "#D55E00", "PC scores": "#CC79A7",
}
FEATURESET_SPARE = ["#56B4E9", "#F0E442", "#000000", "#999999", "#7570B3"]
FEATURESET_COLORS = {}
for _i, _fs in enumerate(FEATURESET_ORDER):
    FEATURESET_COLORS[_fs] = FEATURESET_FIXED.get(
        _fs, FEATURESET_SPARE[_i % len(FEATURESET_SPARE)])

# One colour per dataset, for the feature-importance comparison at the end
# where colour encodes dataset.
DATASET_COLORS = {"perera": "tab:blue", "gesmundo": "tab:orange",
                  "ahneman": "tab:green"}

TICK_ANGLE = 40       # shared x-tick rotation
TICK_FONTSIZE = 8     # 21 tick labels across three datasets, two of them long
CHART_SIZE = (13, 5)  # wide enough that the seven sections do not collide

### 5-fold CV by feature set

One bar per feature set, each in its own colour, with a slight gap and a bracket
per dataset. `DATASET_NAMES`, `TITLE` and `YLABEL` are editable.

In [ ]:
# ---- Editable ----
DATASET_NAMES = {name: info_by_dataset[name]["label"] for name in DATASETS}
TITLE = "Pooled 5-fold CV R$^2$ by feature set"
YLABEL = "pooled R$^2$"
# -------------------

METRIC = "r2"
METHOD = "kfold"

plt.close("all")
datasets = list(DATASET_NAMES)
positions, bounds = grouped_bar_positions(len(datasets), len(FEATURESET_ORDER),
                                          bar_width=0.8, group_gap=0.9)
bar_colors = [FEATURESET_COLORS[fs] for fs in FEATURESET_ORDER]

fig, ax = plt.subplots(figsize=CHART_SIZE)
all_values = []
for group_pos, dataset in zip(positions, datasets):
    part = combined[(combined.dataset == dataset) & (combined.method == METHOD)]
    values = part.set_index("featureset").reindex(FEATURESET_ORDER)[METRIC].to_numpy(dtype=float)
    all_values.append(values)
    bars = ax.bar(group_pos, values, width=0.78, edgecolor="white", linewidth=0.6,
                  color=bar_colors)
    label_bars(ax, bars, values, fontsize=7)

ax.set_xticks(np.concatenate(positions))
ax.set_xticklabels(FEATURESET_ORDER * len(datasets), rotation=TICK_ANGLE,
                   ha="right", fontsize=TICK_FONTSIZE)
draw_group_brackets(ax, bounds, [DATASET_NAMES[d] for d in datasets])

handles = [plt.Rectangle((0, 0), 1, 1, color=FEATURESET_COLORS[fs]) for fs in FEATURESET_ORDER]
ax.legend(handles, FEATURESET_ORDER, title="Feature set", fontsize=8, title_fontsize=8,
          frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1.0))
ax.set(xlabel="", ylabel=YLABEL)
ax.set_title(TITLE, fontsize=11)
set_bar_ylim(ax, np.concatenate(all_values))
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.set_axisbelow(True)
fig.tight_layout()
fig.subplots_adjust(bottom=0.42)

fig_kfold = fig
display(fig_kfold)
plt.close(fig_kfold)

### Matched IID against LOLO

Two bars per feature set. The gap between them is the cost of meeting an unseen
ligand: the matched control has LOLO's fold geometry with every group present in
training, so the pair differs only in that one respect.

In [ ]:
# ---- Editable ----
DATASET_NAMES2 = {name: info_by_dataset[name]["label"] for name in DATASETS}
TITLE2 = "Matched IID vs LOLO R$^2$ by feature set"
YLABEL2 = "pooled R$^2$"
# --------------------

METRIC2 = "r2"
METHODS2 = ["iid_matched", "lolo"]      # draw order = legend order

plt.close("all")
datasets2 = list(DATASET_NAMES2)
bar_width = 0.36
pair_width = bar_width * len(METHODS2)
positions2, bounds2 = grouped_bar_positions(len(datasets2), len(FEATURESET_ORDER),
                                            bar_width=pair_width, group_gap=0.9)

fig, ax = plt.subplots(figsize=CHART_SIZE)
all_values2 = []
for i, method in enumerate(METHODS2):
    xs, heights = [], []
    for group_pos, dataset in zip(positions2, datasets2):
        part = combined[(combined.dataset == dataset) & (combined.method == method)]
        values = part.set_index("featureset").reindex(FEATURESET_ORDER)[METRIC2].to_numpy(dtype=float)
        xs.append(group_pos - pair_width / 2 + bar_width * (i + 0.5))
        heights.append(values)
    xs, heights = np.concatenate(xs), np.concatenate(heights)
    all_values2.append(heights)
    bars = ax.bar(xs, heights, width=bar_width * 0.92, edgecolor="white", linewidth=0.6,
                  color=METHOD_COLORS[method], label=method_label(method))
    label_bars(ax, bars, heights, fontsize=6)

ax.set_xticks(np.concatenate(positions2))
ax.set_xticklabels(FEATURESET_ORDER * len(datasets2), rotation=TICK_ANGLE,
                   ha="right", fontsize=TICK_FONTSIZE)
draw_group_brackets(ax, bounds2, [DATASET_NAMES2[d] for d in datasets2])
ax.legend(title="Evaluation method", fontsize=8, title_fontsize=8, frameon=False,
          loc="upper left", bbox_to_anchor=(1.01, 1.0))
ax.set(xlabel="", ylabel=YLABEL2)
ax.set_title(TITLE2, fontsize=11)
set_bar_ylim(ax, np.concatenate(all_values2))
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.set_axisbelow(True)
fig.tight_layout()
fig.subplots_adjust(bottom=0.42)

fig_methods = fig
display(fig_methods)
plt.close(fig_methods)

### Every dataset's ligands in one Kraken space

Which parts of the reference the three screens actually cover, and where they
overlap. The grey cloud is the same 1,223-ligand reference behind every
per-dataset figure.

In [ ]:
from gpc.report import dataset_ligands, kraken_population

# ---- Editable ----
TITLE3 = "Screened ligands in the Kraken feature space"
MARKERS = {"perera": ("o", "#c0392b", 110), "ahneman": ("*", "#8e44ad", 220),
           "gesmundo": ("^", "#2ca02c", 90)}
SHOW_LABELS3 = False      # 17 labels in one panel is a thicket; True to try
# -------------------

plt.close("all")
population = kraken_population(ROOT / "datasets" / list(DATASETS)[0] / "inputs")
fig, axes = plt.subplots(1, 2, figsize=(13.5, 6), constrained_layout=True)

for ax, (xcol, ycol, xlabel, ylabel, title) in zip(axes, [
        ("PC1", "PC2", "PC1", "PC2", "Kraken Database"),
        ("vbur_pct_boltz", "vbur_pct_min",
         figures.STYLE["kraken"]["vbur_xlabel"],
         figures.STYLE["kraken"]["vbur_ylabel"], "Buried volume")]):
    ax.scatter(population[xcol], population[ycol], s=18, c="0.78",
               edgecolors="none", zorder=1,
               label=f"Kraken reference ({len(population)})")
    for name in DATASETS:
        used = dataset_ligands(ROOT / "datasets" / name / "inputs", population)
        marker, colour, size = MARKERS.get(name, ("o", "tab:blue", 100))
        ax.scatter(used[xcol], used[ycol], s=size, marker=marker, c=colour,
                   edgecolors="white", linewidths=0.7, zorder=3,
                   label=f"{info_by_dataset[name]['label']} ({len(used)})")
        if SHOW_LABELS3:
            for _, row in used.iterrows():
                ax.annotate(row["name"], (row[xcol], row[ycol]), xytext=(6, 6),
                            textcoords="offset points", fontsize=8)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.spines[["top", "right"]].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_linewidth(1.8)
        ax.spines[side].set_color("0.15")
    ax.tick_params(width=1.8, length=6, labelsize=11, color="0.15")

axes[0].set_ylim(*figures.STYLE["kraken"]["pc_ylim"])
axes[0].legend(frameon=False, fontsize=10, loc="upper left")
fig.suptitle(TITLE3, fontsize=15, fontweight="bold")

fig_space = fig
display(fig_space)
plt.close(fig_space)

### Feature importance across datasets

One panel per feature set, one colour per dataset, sharing a feature axis so
rank and bar length can be read against each other directly.

`Selected-2`, `Selected-5` and `PC scores` use the same descriptor and component
names in every dataset, so their bars line up feature for feature. **`PC top` and
`OHE` do not**: `pc_top` is each reference PCA's top-loaded descriptors (the same
here, since the PCA is shared) and `OHE` is each screen's own ligand identities,
which have nothing in common. Read those two panels as three separate rankings
that happen to share an axis.

Relevances are **not** comparable between datasets or between sections -- each
fit has its own kernel and its own scale. Compare the ordering.

In [ ]:
# ---- Editable ----
# Every section with an explicit descriptor list, plus the PC ones. Taken from
# config.SELECTED_SETS so this follows new sections automatically too.
IMPORTANCE_SETS = [*SELECTED_SETS, "pc_scores", "pc_top"]
TOP_N4 = 8
TITLE4 = "ARD relevance by feature set, across datasets"
FLAT_SPREAD = 0.01         # relative spread below this is not a ranking at all
# -------------------

plt.close("all")
records, unavailable, degenerate = [], [], []
for name, entry in DATASETS.items():
    bundle = ROOT / "datasets" / name / "inputs"
    for model in IMPORTANCE_SETS:
        try:
            # top=None, not top=TOP_N4: the features shown are chosen once,
            # below, across all datasets. Taking each dataset's own top N here
            # would leave a dataset with no bar for a feature that simply
            # missed its cut, which reads as zero relevance rather than as
            # "not in this one's top few".
            ranked = rank_features(ROOT / entry["runs"], bundle, model,
                                   method="lolo", ligand_only=True, top=None)
        except (ValueError, FileNotFoundError) as exc:
            unavailable.append((name, model, str(exc).split(" -- ")[0]))
            continue
        relevance = ranked["median_relevance"].to_numpy(dtype=float)
        # Relative, not absolute: an optimiser that never moved still leaves a
        # little numerical scatter, and what matters is that the scatter is
        # negligible against the value itself.
        spread = float(np.ptp(relevance)) / max(abs(float(np.median(relevance))), 1e-12)
        if len(relevance) > 1 and spread < FLAT_SPREAD:
            # Every lengthscale sat where it started: the fit did not identify
            # any feature. Common when the section is wide and the screen has
            # few groups -- Ahneman's four ligands over pc_top's 40 descriptors.
            degenerate.append((name, model, float(relevance[0]), len(relevance)))
        for _, row in ranked.iterrows():
            records.append({"dataset": name, "model": model,
                            "feature": row["feature"].replace("num__", "").replace("cat__", ""),
                            "relevance": row["median_relevance"]})

for name, model, reason in unavailable:
    print(f"{name}/{model}: {reason}")
for name, model, value, n in degenerate:
    print(f"NOTE {name}/{model}: all {n} relevances are {value:.3f} -- the ARD "
          f"lengthscales never moved off their prior, so this panel shows no "
          f"ranking for it, only that none was learned.")

if records:
    ranks = pd.DataFrame(records)
    sets = [m for m in IMPORTANCE_SETS if m in set(ranks.model)]
    fig, axes = plt.subplots(len(sets), 1, figsize=(9, 3.1 * len(sets)), squeeze=False)
    for ax, model in zip(axes.ravel(), sets):
        part = ranks[ranks.model == model]
        # Feature order and selection are both from the mean across datasets,
        # so the axis is not whichever dataset happened to load first, and
        # every dataset has a bar for every feature drawn.
        mean_relevance = part.groupby("feature").relevance.mean().sort_values(ascending=False)
        order = mean_relevance.head(TOP_N4).index.tolist()
        names = list(DATASETS)
        height = 0.8 / len(names)
        for i, dataset in enumerate(names):
            sub = part[part.dataset == dataset].set_index("feature").reindex(order)
            y = np.arange(len(order)) + (i - (len(names) - 1) / 2) * height
            ax.barh(y, sub.relevance.to_numpy(dtype=float), height=height * 0.9,
                    color=DATASET_COLORS.get(dataset, "0.5"),
                    label=info_by_dataset[dataset]["label"])
        ax.set_yticks(np.arange(len(order)))
        ax.set_yticklabels(order, fontsize=8)
        ax.invert_yaxis()
        ax.set_xlabel("median ARD relevance (1 / lengthscale)", fontsize=9)
        flat = [n for n, m, _, _ in degenerate if m == model]
        suffix = f"   (no ranking learned for: {', '.join(flat)})" if flat else ""
        ax.set_title(figures.featureset_label(model) + suffix, fontsize=10)
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="x", alpha=0.25, linewidth=0.6)
        ax.set_axisbelow(True)
    axes.ravel()[0].legend(title="Dataset", fontsize=8, title_fontsize=8, frameon=False,
                           loc="lower right")
    fig.suptitle(TITLE4, fontsize=12, fontweight="bold")
    fig.tight_layout()
    fig_importance = fig
    display(fig_importance)
    plt.close(fig_importance)
else:
    print("No ARD lengthscales available -- these runs were fitted with ard: false.")

## Save

Nothing is written until `SAVE = True`. These go to `results/_comparisons/`
rather than into any one dataset's folder, since they belong to none of them.

In [ ]:
SAVE = False
COMPARISON_DIR = ROOT / "results" / "_comparisons"
FORMATS = ["png"]

if SAVE:
    COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
    combined.to_csv(COMPARISON_DIR / "pooled_by_dataset.csv", index=False)
    # globals().get: a section that was never run leaves its name undefined,
    # and that must not break the export.
    for stem in ("fig_kfold", "fig_methods", "fig_space", "fig_importance"):
        fig = globals().get(stem)
        if fig is None:
            continue
        for fmt in FORMATS:
            fig.savefig(COMPARISON_DIR / f"{stem[4:]}.{fmt}", dpi=300, bbox_inches="tight")
    print("saved ->", COMPARISON_DIR)
else:
    print("Preview only. Set SAVE=True when ready.")